# QUANTUM MACHINE LEARNING PROBLEM USING THE WEBSITE PHISHING DATASET

In [5]:
# Import the required library and framework
import numpy as np
import pandas as pd


from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from qiskit import QuantumCircuit, generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as Estimator
from qiskit_algorithms.utils import algorithm_globals
from qiskit_algorithms.optimizers import ADAM, SPSA
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer import AerSimulator


import torch
from torch.utils.data import DataLoader, TensorDataset
import random
from functools import partial


In [6]:
# First we process the data
df = pd.read_csv("data.csv")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
print(f"Missing values: {df.isna().sum().sum()} | Duplicate rows: {df.duplicated().sum()}")
df.head()

Rows: 11,055 | Columns: 32
Missing values: 0 | Duplicate rows: 0


,id,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,...,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,Result
0,1,-1,1,1,1,-1,-1,-1,-1,-1,...,1,1,-1,-1,-1,-1,1,1,-1,-1
1,2,1,1,1,1,1,-1,0,1,-1,...,1,1,-1,-1,0,-1,1,1,1,-1
2,3,1,0,1,1,1,-1,-1,-1,-1,...,1,1,1,-1,1,-1,1,0,-1,-1
3,4,1,0,1,1,1,-1,-1,-1,1,...,1,1,-1,-1,1,-1,1,-1,1,-1
4,5,1,0,-1,1,1,-1,1,1,-1,...,-1,1,-1,-1,0,-1,1,1,1,1


In [7]:
# We process data, choose the features, labela and divide the dataset into training and testing set
feature_cols = [col for col in df.columns if col != 'Result' and col != 'id']
print(feature_cols)
X = df[feature_cols]
y = df["Result"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

['having_IP_Address', 'URL_Length', 'Shortining_Service', 'having_At_Symbol', 'double_slash_redirecting', 'Prefix_Suffix', 'having_Sub_Domain', 'SSLfinal_State', 'Domain_registeration_length', 'Favicon', 'port', 'HTTPS_token', 'Request_URL', 'URL_of_Anchor', 'Links_in_tags', 'SFH', 'Submitting_to_email', 'Abnormal_URL', 'Redirect', 'on_mouseover', 'RightClick', 'popUpWidnow', 'Iframe', 'age_of_domain', 'DNSRecord', 'web_traffic', 'Page_Rank', 'Google_Index', 'Links_pointing_to_page', 'Statistical_report']


In [8]:
# Training the full dataset on classical model
scaler_1 = StandardScaler()
X_train_scaled = scaler_1.fit_transform(X_train)
X_test_scaled = scaler_1.transform(X_test)
model  = LogisticRegression(max_iter = 1000)
model.fit(X_train_scaled, y_train)

acc = accuracy_score(y_test, model.predict(X_test_scaled)) * 100
print(f"Logistic Regression model accuracy: {acc:.2f}%")

Logistic Regression model accuracy: 92.19%


# Because full 30 features are too much for qubits to represent. I'll choose 6 qubits for this problem. I need to pick either the 6 most influential features or PCA to get the 6 aggregated features 

In [9]:
# Option 1: Choose the 6 most influential features
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=X_train.columns)
top_features = importances.sort_values(ascending=False).head(6)
print(top_features)

SSLfinal_State       0.307502
URL_of_Anchor        0.264289
having_Sub_Domain    0.069721
web_traffic          0.066930
Links_in_tags        0.042175
Prefix_Suffix        0.037801
dtype: float64


In [10]:
top6 = top_features.index
X_train_top6 = X_train[top6]
print(type(X_train_top6))
X_test_top6 = X_test[top6]
print(X_train_top6.head(5))

<class 'pandas.DataFrame'>
       SSLfinal_State  URL_of_Anchor  having_Sub_Domain  web_traffic  \
10168              -1             -1                  0            0   
10167              -1              0                  1            1   
2115               -1             -1                  0           -1   
1699               -1             -1                 -1            0   
1379                1              0                  1            1   

       Links_in_tags  Prefix_Suffix  
10168              1             -1  
10167             -1             -1  
2115               0             -1  
1699              -1             -1  
1379              -1             -1  


In [11]:
# Retraining the model on these 6 features
model_2 = LogisticRegression(max_iter=1000)
model_2.fit(X_train_top6, y_train)

acc = accuracy_score(y_test, model_2.predict(X_test_top6)) * 100
print(f"Logistic Regression model accuracy: {acc:.2f}%")

Logistic Regression model accuracy: 91.29%


In [30]:
# Option 2: Use PCA to get the 6 aggregated features

pca = PCA(n_components=6)
X_pca_train = pca.fit_transform(X_train)
X_pca_test  = pca.transform(X_test)
model_3 = LogisticRegression(max_iter=1000)
model_3.fit(X_pca_train, y_train)

acc = accuracy_score(y_test, model_3.predict(X_pca_test)) * 100
print(f"Logistic Regression model accuracy: {acc:.2f}%")

Logistic Regression model accuracy: 90.38%


# Clearly, using 6 most influential features yield better result than using PCA, we will stick to option 1 and focus on only these 6 features

# NOW WE GO TO THE QUANTUM FRAMEWORK BY BUILDING A VQC - VARIATIONAL QUANTUM CLASSIFIER

In [13]:
# We will focus on these 6 features only
num_features = X_train_top6.shape[1]
num_qubits = num_features 

mm_scaler = MinMaxScaler(feature_range=(0, np.pi))

# Define the TensorDataset and divide the dataset into small batch 
X_train_top6_mm = mm_scaler.fit_transform(X_train_top6)
X_test_top6_mm = mm_scaler.transform(X_test_top6)

# First we switch the dataframe object to tensor. By doing this, we can easily divide them into batch of data
print(type(X_train_top6_mm))
X_train_top6_mm = torch.tensor(X_train_top6_mm, dtype=torch.float32)
print(type(X_train_top6_mm))
X_test_top6_mm  = torch.tensor(X_test_top6_mm, dtype=torch.float32)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.long)
print(type(y_train))
y_test  = torch.tensor(y_test.to_numpy(), dtype=torch.long)


<class 'numpy.ndarray'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>


In [14]:
# First define the variables
num_layers = 2
feature_map = ZZFeatureMap(feature_dimension=num_qubits, entanglement='full') # This would be used as our main encoding circuit
observable = SparsePauliOp("I" * (num_qubits - 1) + "Z")
estimator = Estimator()

# Define the encoding circuit

def feature_encoding_circuit(features, num_layers, lam=np.pi):
    qc = QuantumCircuit(num_qubits)
    for i in range(num_qubits):
        qc.h(i)
    for _ in range(num_layers):
        # Apply the rotation x layer
        for i in range(num_qubits):
            qc.rx(lam*features[i], i)
        # Apply the controlled not layer
        for i in range(num_qubits-1):
            qc.cx(i, i+1)
    return qc

def data_encoding_circuit(features, lam=np.pi):
    qc = QuantumCircuit(num_qubits)
    for i in range(num_qubits):
        qc.ry(lam*features[i], i)
    return qc

# Define the ansatz
def ansatz(params, qc):
    num_feature = num_qubits
    for l in range(num_layers):
        for i in range(num_feature):
            qc.rx(params[l,i],i)
        for i in range(num_feature-1):
            qc.cx(i,i+1)
        if num_feature>1:
            qc.cx(num_feature-1,0)
    return qc

ansatz_model  = RealAmplitudes(num_qubits= num_qubits, reps=num_layers-1, entanglement='full') # This would be used as our main ansatz
full_circuit_template = feature_map.compose(ansatz_model).decompose()

# Define our VQC
def quantum_classifier_circuit(params, features):
    """Full quantum circuit: encoding + ansatz + measurement."""
    param_dict = dict(zip(full_circuit_template.parameters, np.append(features, params)))
    qc = full_circuit_template.assign_parameters(param_dict)
    # Measure the expectation value of Pauli Z on the first qubit
    job = estimator.run([(qc, observable)])
    result = job.result()

    # Extract expectation value
    expval = result[0].data.evs
    return expval

def weighted_square_loss(labels, predictions, sample_weights):
    """Mean squared loss, weighted per-sample by class."""
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    sample_weights = np.asarray(sample_weights)
    squared_errors = (labels - predictions) ** 2
    return np.average(squared_errors, weights=sample_weights)


def square_loss(labels, predictions):
    """Calculates the mean squared loss."""
    return np.mean((labels - predictions)**2)
# Define the cost function
def cost_function(params, features_batch, labels_batch, class_weight=None):
    """Cost function to be minimized."""
    predictions = [quantum_classifier_circuit(params, f) for f in features_batch]
    if class_weight is not None:
        sample_weights = labels_batch.map(class_weight).values
        cost = weighted_square_loss(labels_batch, predictions, sample_weights)
    else:
        cost = np.mean((np.asarray(labels_batch) - np.asarray(predictions)) ** 2)

    return [cost, predictions]

/var/folders/m9/3nzfzfxj289195w21ntjp4m40000gn/T/ipykernel_29830/1490273371.py:3: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=num_qubits, entanglement='full') # This would be used as our main encoding circuit
/var/folders/m9/3nzfzfxj289195w21ntjp4m40000gn/T/ipykernel_29830/1490273371.py:40: DeprecationWarning: The class ``qiskit.circuit.library.n_local.real_amplitudes.RealAmplitudes`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.real_amplitudes instead.
  ansatz_model  = RealAmplitudes(num_qubits= num_qubits, reps=num_layers-1, entanglement='full') # This would be used as our main ansatz


In [15]:
# Now is the training step for our VQC
# Define the random initial parameters
SEED = 42
random.seed(SEED) 
np.random.seed(SEED)
algorithm_globals.random_seed = SEED  

param_shape = (num_layers, num_qubits)
initial_params = np.random.uniform(low=0, high=2 * np.pi, size=param_shape)
flat_initial_params = initial_params.flatten()

# Define the optimizer Adam and SPSA
optimizer = ADAM(maxiter=50, lr=0.05)
optimizer_2 = SPSA(maxiter=300)
cost_history = []



dataset = TensorDataset(X_train_top6_mm, y_train)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)
CLASS_WEIGHT = {-1: 3.0, 1: 1.0} 


In [16]:
# Define the objective / loss function that optimizer aims to minimize

def loss_fn(flat_p):
    p = flat_p.reshape(param_shape)
    
    index = random.randint(1,len(train_loader)-1)
    batch_X, batch_y = next(
        batch for i, batch in enumerate(train_loader)
        if i == index
    )

    # Compute cost over the training set
    batch_X = batch_X.detach().cpu().numpy()
    batch_y = pd.Series(batch_y.detach().cpu().numpy())
  
    cost, predictions = cost_function(p, batch_X, batch_y,  class_weight=CLASS_WEIGHT)


    predictions = np.sign(predictions)
    cost_history.append(cost)
    
    # Compute current training accuracy for progress tracking
    acc = np.mean(predictions == batch_y)
    
    if len(cost_history)%5 == 0 or len(cost_history) == 1:
        print(f"Step {len(cost_history)} - Cost: {cost:.4f} - Train Acc: {acc:.4f}")
    
    return cost


# Actual training step, we use SPSA
print("Starting training...")
optimizer_2.calibrate(loss_fn, flat_initial_params)
result_2 = optimizer_2.minimize(loss_fn, flat_initial_params)

# Retrieve optimal parameters reshaped to 3D after training completed
optimal_params = result_2.x.reshape(param_shape)
print("Training finished.")


Starting training...
Step 1 - Cost: 1.3247 - Train Acc: 0.2812
Step 5 - Cost: 1.3954 - Train Acc: 0.2500
Step 10 - Cost: 1.4330 - Train Acc: 0.2031
Step 15 - Cost: 1.5637 - Train Acc: 0.1719
Step 20 - Cost: 1.2876 - Train Acc: 0.2500
Step 25 - Cost: 1.5899 - Train Acc: 0.2969
Step 30 - Cost: 1.4153 - Train Acc: 0.2031
Step 35 - Cost: 1.6036 - Train Acc: 0.2500
Step 40 - Cost: 1.2871 - Train Acc: 0.3125
Step 45 - Cost: 1.3496 - Train Acc: 0.1875
Step 50 - Cost: 1.6188 - Train Acc: 0.1719
Step 55 - Cost: 1.5283 - Train Acc: 0.1406
Step 60 - Cost: 1.7107 - Train Acc: 0.2344
Step 65 - Cost: 1.4325 - Train Acc: 0.3594
Step 70 - Cost: 1.3582 - Train Acc: 0.2500
Step 75 - Cost: 1.3324 - Train Acc: 0.2969
Step 80 - Cost: 1.4352 - Train Acc: 0.2188
Step 85 - Cost: 1.3499 - Train Acc: 0.2656
Step 90 - Cost: 1.4687 - Train Acc: 0.2344
Step 95 - Cost: 1.2082 - Train Acc: 0.2812
Step 100 - Cost: 1.1321 - Train Acc: 0.3438
Step 105 - Cost: 1.3016 - Train Acc: 0.3594
Step 110 - Cost: 0.9903 - Train A

In [17]:

predictions = np.sign([quantum_classifier_circuit(optimal_params, f) for f in X_test_top6_mm])

acc = np.mean(predictions == pd.Series(y_test.detach().cpu().numpy()))
print(f"VQC model accuracy: {acc * 100}%")


VQC model accuracy: 73.83177570093457%


In [18]:
conf_matrix = confusion_matrix(pd.Series(y_test.detach().cpu().numpy()), predictions)
print("Confusion Matrix:")
print(conf_matrix)

Confusion Matrix:
[[ 613  815]
 [  53 1836]]


In [19]:
class_report = classification_report(y_test.detach().cpu().numpy(), predictions)
print("Classification Report:")
print(class_report)

Classification Report:
              precision    recall  f1-score   support

          -1       0.92      0.43      0.59      1428
           1       0.69      0.97      0.81      1889

    accuracy                           0.74      3317
   macro avg       0.81      0.70      0.70      3317
weighted avg       0.79      0.74      0.71      3317



# NOW WE INTRODUCE NOISE TO OUR SIMULATION

In [20]:
prob_1q = 0.01   # 0.1% error per single-qubit gate
prob_2q = 0.01    # 1% error per two-qubit gate (e.g. CX) — try 0.02 for "noisier" scenario too

error_1q = depolarizing_error(prob_1q, 1)
error_2q = depolarizing_error(prob_2q, 2)

noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(error_1q, ['rx', 'ry', 'rz', 'u1', 'u2', 'u3'])  
noise_model.add_all_qubit_quantum_error(error_2q, ['cx'])


noisy_backend = AerSimulator(noise_model=noise_model)
noisy_estimator = Estimator.from_backend(noisy_backend)
pass_manager = generate_preset_pass_manager(optimization_level=1, backend=noisy_backend)
noisy_circuit_template = pass_manager.run(full_circuit_template)

In [21]:
def quantum_classifier_circuit_noisy(params, features):
    """Same circuit, run through the noisy estimator instead."""
    param_dict = dict(zip(noisy_circuit_template.parameters, np.append(features, params)))
    qc = noisy_circuit_template.assign_parameters(param_dict)
    job = noisy_estimator.run([(qc, observable)])
    result = job.result()
    expval = result[0].data.evs
    return expval

In [22]:
test_features = X_test_top6_mm[0]
ideal_val = quantum_classifier_circuit(optimal_params, test_features)
noisy_val = quantum_classifier_circuit_noisy(optimal_params, test_features)
print(f"Ideal: {ideal_val:.4f}, Noisy: {noisy_val:.4f}")

Ideal: -0.3206, Noisy: -0.1321


In [23]:

predictions_1 = np.sign([quantum_classifier_circuit_noisy(optimal_params, f) for f in X_test_top6_mm])

acc = np.mean(predictions_1 == pd.Series(y_test.detach().cpu().numpy()))
print(f"VQC model with errors accuracy: {acc * 100}%")

VQC model with errors accuracy: 75.82152547482664%


In [24]:
conf_matrix = confusion_matrix(pd.Series(y_test.detach().cpu().numpy()), predictions_1)
print("Confusion Matrix:")
print(conf_matrix)

Confusion Matrix:
[[ 697  731]
 [  71 1818]]


In [25]:
class_report = classification_report(y_test.detach().cpu().numpy(), predictions_1)
print("Classification Report:")
print(class_report)

Classification Report:
              precision    recall  f1-score   support

          -1       0.91      0.49      0.63      1428
           1       0.71      0.96      0.82      1889

    accuracy                           0.76      3317
   macro avg       0.81      0.73      0.73      3317
weighted avg       0.80      0.76      0.74      3317



In [26]:
noise_levels_2q = [0.0, 0.005, 0.01, 0.0125, 0.015, 0.0175, 0.02]
results = []



def quantum_classifier_circuit_noisy(params, features, noisy_circuit_template, noisy_estimator):
    """Same circuit, run through the noisy estimator instead."""
    param_dict = dict(zip(noisy_circuit_template.parameters, np.append(features, params)))
    qc = noisy_circuit_template.assign_parameters(param_dict)        
    job = noisy_estimator.run([(qc, observable)])
    result = job.result()
    expval = result[0].data.evs
    return expval

for noise_levels in noise_levels_2q:
    prob_1q = noise_levels
    prob_2q = noise_levels

    error_1q = depolarizing_error(prob_1q, 1)
    error_2q = depolarizing_error(prob_2q, 2)

    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(error_1q, ['rx', 'ry', 'rz', 'u1', 'u2', 'u3'])  
    noise_model.add_all_qubit_quantum_error(error_2q, ['cx'])


    noisy_backend = AerSimulator(noise_model=noise_model)
    noisy_estimator = Estimator.from_backend(noisy_backend)
    pass_manager = generate_preset_pass_manager(optimization_level=1, backend=noisy_backend)
    noisy_circuit_template = pass_manager.run(full_circuit_template)

    print(f"Predicting using circuits with {noise_levels} percentage of noise")
    predictions_2 = np.sign([quantum_classifier_circuit_noisy(optimal_params, f, noisy_circuit_template, noisy_estimator) for f in X_test_top6_mm])
    print(np.unique(predictions_2, return_counts=True))
    acc = np.mean(predictions_2 == pd.Series(y_test.detach().cpu().numpy()))
    results.append((noise_levels, acc))
    print(f"VQC model with errors {noise_levels} accuracy: {acc * 100}%")



Predicting using circuits with 0.0 percentage of noise
(array([-1.,  1.]), array([ 666, 2651]))
VQC model with errors 0.0 accuracy: 73.83177570093457%
Predicting using circuits with 0.005 percentage of noise
(array([-1.,  1.]), array([ 677, 2640]))
VQC model with errors 0.005 accuracy: 74.10310521555623%
Predicting using circuits with 0.01 percentage of noise
(array([-1.,  1.]), array([ 768, 2549]))
VQC model with errors 0.01 accuracy: 75.82152547482664%
Predicting using circuits with 0.0125 percentage of noise
(array([-1.,  1.]), array([ 811, 2506]))
VQC model with errors 0.0125 accuracy: 75.8516731986735%
Predicting using circuits with 0.015 percentage of noise
(array([-1.,  1.]), array([ 851, 2466]))
VQC model with errors 0.015 accuracy: 77.05758215254748%
Predicting using circuits with 0.0175 percentage of noise
(array([-1.,  1.]), array([ 737, 2580]))
VQC model with errors 0.0175 accuracy: 73.86192342478142%
Predicting using circuits with 0.02 percentage of noise
(array([-1.,  1.]

# Now we will use Zero Noise Extrapolation to mitigate the effect of noise

In [28]:
def get_noisy_estimator(prob_2q_base, scale_factor):
    """Build a noisy estimator with noise scaled by scale_factor (for ZNE)."""
    prob_1q = prob_2q_base * scale_factor
    prob_2q = prob_2q_base * scale_factor
    error_1q = depolarizing_error(min(prob_1q, 0.75), 1)  # clip to valid probability range
    error_2q = depolarizing_error(min(prob_2q, 0.75), 2)
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(error_1q, ['rx', 'ry', 'rz', 'u1', 'u2', 'u3'])
    noise_model.add_all_qubit_quantum_error(error_2q, ['cx'])
    backend = AerSimulator(noise_model=noise_model)
    estimator = Estimator.from_backend(backend)
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    template = pm.run(full_circuit_template)
    return estimator, template


def zne_predict(params, features, base_noise, scale_factors=[1, 1.2, 1.3]):
    """Estimate zero-noise value by linear extrapolation across noise-scaled runs."""
    expvals = []
    for sf in scale_factors:
        est, tmpl = get_noisy_estimator(base_noise, sf)
        val = quantum_classifier_circuit_noisy(params, features, tmpl, est)
        expvals.append(val)
    # Linear fit: expval vs scale_factor, extrapolate to scale_factor=0
    coeffs = np.polyfit(scale_factors, expvals, deg=1)
    zero_noise_estimate = coeffs[-1]  # intercept at x=0
    return zero_noise_estimate



zne_expvals = [zne_predict(optimal_params, f, 0.0175) for f in X_test_top6_mm]
zne_predictions = np.sign(zne_expvals)
zne_acc = np.mean(zne_predictions == pd.Series(y_test.detach().cpu().numpy()))
print(f"ZNE-mitigated accuracy: {zne_acc:.4f}")

ZNE-mitigated accuracy: 0.7585
